In [1]:
import time
from pyspark.sql import SparkSession 
from pyspark.sql import functions as F 
from pyspark.sql import types as T
from kafka import KafkaProducer

spark = SparkSession \
    .builder \
    .master("local") \
    .appName("ex6_Real-time_Review") \
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0') \
    .getOrCreate()
   


:: loading settings :: url = jar:file:/usr/local/lib/python3.10/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/developer/.ivy2/cache
The jars for the packages stored in: /home/developer/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-071e7555-a9c0-427d-936a-df1e8d802568;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.0 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.9.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
downloading https://repo1.maven.org/maven2/org/apache/spa

In [3]:
#Set up a Kafka producer
producer = KafkaProducer(bootstrap_servers='course-kafka:9092', value_serializer=lambda v: v.encode('utf-8'))

In [4]:
stream_df = spark \
.readStream \
.format("kafka") \
.option("kafka.bootstrap.servers", "course-kafka:9092") \
.option("subscribe", "gps-user-review-source") \
.option("startingOffsets", "earliest") \
.load() \
.select(F.col("value").cast("string"))


In [5]:
#Parse the JSON Data
json_schema = T.StructType([
    T.StructField('application_name', T.StringType()),
    T.StructField('translated_review', T.StringType()),
    T.StructField('sentiment_rank', T.StringType()),
    T.StructField('sentiment_polarity', T.StringType()),
    T.StructField('sentiment_subjectivity', T.StringType())
])

In [6]:
#parse data
parsed_df = stream_df \
.withColumn('parsed_json', F.from_json(F.col("value"), json_schema)) \
.select(F.col('parsed_json.*'))

In [7]:
#Load and Cache Static Data:
static_data_df = spark.read.parquet("s3a://pyspark/data/source/google_apps/")
static_data_df.cache()

26/08/14 00:06:20 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


DataFrame[application_name: string, category: string, rating: string, reviews: float, size: string, num_of_installs: double, price: double, age_limit: bigint, genres: string, version: string]

In [9]:
joined_df = parsed_df.groupBy("application_name")\
.agg(F.sum(F.when(F.col("sentiment_rank")==1,1).otherwise(0)).alias("num_of_positive_sentiments"),
            F.sum(F.when(F.col("sentiment_rank")==-1,1).otherwise(0)).alias("num_of_negative_sentiments"),
            F.sum(F.when(F.col("sentiment_rank")==0,1).otherwise(0)).alias("num_of_neutral_sentiments"),
            F.avg("sentiment_polarity").alias("avg_sentiment_polarity"),
            F.avg("sentiment_subjectivity").alias("avg_sentiment_subjectivity")
            )\
.join(static_data_df, on = 'application_name', how = "left")

fieldNames() → מחזיר את שמות העמודות.
F.col() → הופך שם של עמודה לעמודה ש-Spark יודע לעבוד איתה.
map(...) → מבצע את ההמרה הזו על כל העמודות.

In [10]:
fields_list = joined_df.schema.fieldNames()
fields_as_cols = list(map(lambda col_name: F.col(col_name), fields_list))

In [11]:
#Converting Data to JSON Format
json_df = joined_df.select(F.to_json(F.struct(*fields_as_cols)).alias("value"))


In [13]:
#Writing the Stream to Kafka

query = json_df\
.writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "course-kafka:9092") \
    .option("topic", "gps-with-reviews") \
    .option("checkpointLocation", 's3a://pyspark/checkpoints/ex6/review_calculation') \
    .outputMode("update") \
    .start()

26/08/14 00:10:24 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/08/14 00:10:25 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/08/14 00:10:27 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/08/14 00:10:35 WARN NetworkClient: [Producer clientId=producer-1] Error while fetching metadata with correlation id 1 : {gps-with-reviews=LEADER_NOT_AVAILABLE}
26/08/14 00:10:35 WARN NetworkClient: [Producer clientId=producer-1] Error while fetching metadata with correlation id 4 : {gps-with-reviews=LEADER_NOT_AVAILABLE}


In [18]:
query.awaitTermination()
static_data_df.unpersist()

DataFrame[application_name: string, category: string, rating: string, reviews: float, size: string, num_of_installs: double, price: double, age_limit: bigint, genres: string, version: string]

In [15]:
query.stop()

In [ ]:
spark.stop()
producer.close()

26/08/14 00:16:33 WARN StateStore: Error running maintenance thread
java.lang.IllegalStateException: SparkEnv not active, cannot do maintenance on StateStores
	at org.apache.spark.sql.execution.streaming.state.StateStore$.doMaintenance(StateStore.scala:600)
	at org.apache.spark.sql.execution.streaming.state.StateStore$.$anonfun$startMaintenanceIfNeeded$1(StateStore.scala:586)
	at org.apache.spark.sql.execution.streaming.state.StateStore$MaintenanceTask$$anon$1.run(StateStore.scala:446)
	at java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:515)
	at java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:305)
	at java.base/java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask.run(ScheduledThreadPoolExecutor.java:305)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1128)
	at java.base/java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:628)
	at java.base/java.lang.Thread.

In [72]:
spark.sparkContext.getConf().getAll()

[('spark.master', 'local'),
 ('spark.driver.host', '7e62a4cd2411'),
 ('spark.driver.memory', '4g'),
 ('spark.app.submitTime', '1786636951306'),
 ('spark.executor.id', 'driver'),
 ('spark.sql.warehouse.dir',
  'file:/home/developer/projects/spark-course-python/spark_course_python/lab6/spark-warehouse'),
 ('spark.driver.extraJavaOptions',
  '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-op

In [77]:
print(id(spark))

spark.stop()

builder = SparkSession.builder \
    .master("local") \
    .appName("test") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0"
    )

spark = builder.getOrCreate()

print(id(spark))
print(spark.sparkContext.getConf().get("spark.jars.packages"))

137009943342208


Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: org.apache.spark.SparkException: Only one SparkContext should be running in this JVM (see SPARK-2243).The currently running SparkContext was created at:
org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:62)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:490)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.ConstructorCommand